# ISOM 839 · Session 6 — Integer Optimization I: The Power of Yes/No Decisions

**Prescriptive Analytics: Modeling & Optimization · Suffolk University · Prof. Hasan Arslan**

Last week we routed flow through a network we took as given. Tonight we decide what the network **is**. Open the warehouse or don't. Pay the fixed cost or don't. Launch the project or kill it. That needs a variable that can only be 0 or 1 — one `vtype` change that turns the optimizer from a calculator into a strategist, and makes the problem exponentially harder to solve.

In [ ]:
%pip install -q gurobipy
import gurobipy as gp
from gurobipy import GRB
print('gurobipy ready:', gp.gurobi.version())

---
## Part 1 — Facility location: the fixed-charge trick (tonight's lab)

**Five candidate warehouses, eight customer zones.** Each warehouse costs a hefty fixed amount per year to open and has a capacity; each customer must be served. Shipping costs are dollars per unit per week (we annualize ×52). Which warehouses open, and who serves whom?

Two variable families live in one model:
- `open[w]` — **binary**: 1 if warehouse *w* opens
- `x[w,c]` — **continuous**: units shipped from *w* to customer *c* per week

**The link:** shipments out of *w* ≤ capacity[w] × open[w]. Closed warehouse → zero flow. Note the "M" here is the capacity, not an arbitrary huge number.

In [ ]:
warehouses = {  # name: (fixed cost $/yr, weekly capacity)
    'Worcester': (42000, 900), 'Hartford': (38000, 800), 'Albany': (31000, 700),
    'Providence': (35000, 650), 'Manchester': (29000, 600),
}
customers = {'Boston': 310, 'Cambridge': 140, 'Springfield': 120, 'New Haven': 170,
             'Portland': 130, 'Burlington': 90, 'Nashua': 110, 'Lowell': 150}   # weekly demand

ship = {  # $ per unit per week, warehouse -> customer
    'Worcester':  {'Boston': 4, 'Cambridge': 4, 'Springfield': 7, 'New Haven': 6, 'Portland': 6, 'Burlington': 9, 'Nashua': 6, 'Lowell': 5},
    'Hartford':   {'Boston': 3, 'Cambridge': 7, 'Springfield': 3, 'New Haven': 3, 'Portland': 11, 'Burlington': 3, 'Nashua': 3, 'Lowell': 8},
    'Albany':     {'Boston': 11, 'Cambridge': 12, 'Springfield': 4, 'New Haven': 6, 'Portland': 10, 'Burlington': 6, 'Nashua': 10, 'Lowell': 9},
    'Providence': {'Boston': 5, 'Cambridge': 9, 'Springfield': 6, 'New Haven': 6, 'Portland': 7, 'Burlington': 11, 'Nashua': 12, 'Lowell': 9},
    'Manchester': {'Boston': 6, 'Cambridge': 4, 'Springfield': 8, 'New Haven': 11, 'Portland': 5, 'Burlington': 3, 'Nashua': 2, 'Lowell': 6},
}
annual = {(w, c): ship[w][c] * 52 for w in warehouses for c in customers}
print(f'demand {sum(customers.values())} units/week  vs  capacity if all open {sum(v[1] for v in warehouses.values())}')

In [ ]:
fl = gp.Model('facility_location')
opened = fl.addVars(warehouses.keys(), vtype=GRB.BINARY, name='open')          # the strategic decision
x      = fl.addVars(annual.keys(), name='x')                                  # the operational decision

fl.setObjective(gp.quicksum(warehouses[w][0] * opened[w] for w in warehouses) + x.prod(annual), GRB.MINIMIZE)
fl.addConstrs((x.sum('*', c) == customers[c] for c in customers), 'serve')                    # everyone is served
fl.addConstrs((x.sum(w, '*') <= warehouses[w][1] * opened[w] for w in warehouses), 'link')   # the fixed-charge link
fl.optimize()                                                                                 # watch the log

print(f'\nAnnual cost: ${fl.ObjVal:,.0f}')
print('Open:', [w for w in warehouses if opened[w].X > 0.5])
for (w, c), v in x.items():
    if v.X > 1e-6:
        print(f'  {w:10s} -> {c:11s} {v.X:4.0f}/week')

Two warehouses open — Hartford takes Boston, Springfield, New Haven and Burlington; Manchester takes Cambridge, Portland, Nashua and Lowell — and three stay on paper. Does the geography make sense? Boston is served from Hartford at $3, not from Worcester at $4, because Worcester's $42,000 opening cost buys nothing a cheaper pair cannot do.

### 💬 Discussion: read the log
Scroll up through the solver log. Find three things: the **first incumbent** (a feasible plan), the **best bound** (what the relaxation says is the floor), and the **Gap** closing to 0.00%. That is branch & bound finishing its argument — "no unexplored branch can beat this."

---
## Part 2 — Relax and compare: the price of reality

Change `vtype` to continuous and re-solve. The LP will happily open 0.32 of a warehouse — cheaper, and impossible.

In [ ]:
for w in warehouses:
    opened[w].VType = GRB.CONTINUOUS
fl.Params.OutputFlag = 0
fl.optimize()
lp_value = fl.ObjVal
print(f'LP relaxation: ${fl.ObjVal:,.0f}   open = ' + ', '.join(f'{w} {opened[w].X:.2f}' for w in warehouses if opened[w].X > 1e-6))

for w in warehouses:
    opened[w].VType = GRB.BINARY
fl.optimize()
print(f'Integer plan:  ${fl.ObjVal:,.0f}   open = ' + ', '.join(w for w in warehouses if opened[w].X > 0.5))
print(f'The price of reality: ${fl.ObjVal - lp_value:,.0f} per year')

The relaxation pays 86% of Hartford's fixed cost and 32% of Worcester's — fractions of buildings. The gap between the two numbers is the **integrality gap**: what it costs that warehouses come in whole units. Branch & bound's job is to close that gap by splitting on fractional variables and bounding each branch with its own relaxation.

---
## Part 3 — Logic as algebra

Once you have binaries, business rules become linear constraints:

| Rule in English | Constraint |
|---|---|
| Open at most two warehouses | `open.sum() <= 2` |
| If Albany opens, Hartford must too (if-then) | `open['Albany'] <= open['Hartford']` |
| Worcester or Providence, not both (either/or) | `open['Worcester'] + open['Providence'] <= 1` |
| Boston must be served from one warehouse only (single sourcing) | add binary `assign[w,c]`, `x[w,c] <= demand[c] * assign[w,c]`, `assign.sum('*', c) == 1` |

Add the first rule and see what it costs:

In [ ]:
at_most_two = fl.addConstr(opened.sum() <= 2, 'at_most_two')
fl.optimize()
print(f'At most two open: ${fl.ObjVal:,.0f}  ->', [w for w in warehouses if opened[w].X > 0.5])

fl.remove(at_most_two)
must_use_albany = fl.addConstr(opened['Albany'] == 1, 'must_use_albany')     # politics: Albany opens
fl.optimize()
print(f'Albany forced open: ${fl.ObjVal:,.0f}  ->', [w for w in warehouses if opened[w].X > 0.5])
fl.remove(must_use_albany); fl.optimize()

The first rule costs nothing — the optimum already opened two. The second is a **political constraint with a price tag**, and the model just told you exactly what it is. That is the single most useful sentence a modeler can say in a meeting.

---
## Part 4 — The computational cliff

Five binaries have 32 combinations; you could enumerate them by hand. Thirty have a billion. Sixty have more combinations than there are grains of sand on Earth. Branch & bound does not enumerate — it prunes with bounds — but there are no guarantees, which is why **the MIP gap** in the log is the number a professional reads first, and why tight "M" values matter: a loose M makes the relaxation useless and the tree enormous.

In [ ]:
import random, time
random.seed(839)
n = 60
value  = [random.randint(10, 100) for _ in range(n)]
weight = [random.randint(5, 50) for _ in range(n)]
kp = gp.Model('knapsack_60'); kp.Params.OutputFlag = 0
take = kp.addVars(n, vtype=GRB.BINARY, name='take')
kp.setObjective(gp.quicksum(value[i] * take[i] for i in range(n)), GRB.MAXIMIZE)
kp.addConstr(gp.quicksum(weight[i] * take[i] for i in range(n)) <= sum(weight) // 3, 'capacity')
t0 = time.time(); kp.optimize()
print(f'{2**n:.2e} combinations · solved to proven optimality in {time.time() - t0:.2f}s · value {kp.ObjVal:.0f} · gap {kp.MIPGap:.1%}')

---
# Homework #5 — Project-Selection Knapsack (Binary IP) 💼

**On Canvas: this notebook completed + your 3–4 sentence answer in the text box.** Due date and rubric on the Canvas assignment; late window per the syllabus.

Your division has **$10M of capital** and **12 candidate projects**. Each has an investment cost, an expected NPV, and a staffing need; the division has **60 staff-months** available this year. Two rules from the steering committee:

- **Project 7 (Demand forecasting engine) requires Project 3 (Core ERP upgrade)** — if-then.
- **Projects 5 and 9 (the two retail expansions) are mutually exclusive** — either/or.

| # | Project | Cost $M | NPV $M | Staff-months |
|---|---|---|---|---|
| 1 | Warehouse automation | 2.4 | 3.9 | 14 |
| 2 | Customer data platform | 1.8 | 2.6 | 9 |
| 3 | Core ERP upgrade | 3.1 | 3.4 | 16 |
| 4 | Mobile app relaunch | 0.9 | 1.5 | 7 |
| 5 | Boston retail expansion | 2.7 | 4.1 | 12 |
| 6 | Supplier portal | 0.6 | 0.8 | 4 |
| 7 | Demand forecasting engine | 1.2 | 2.3 | 8 |
| 8 | Energy retrofit | 1.5 | 1.7 | 3 |
| 9 | Hartford retail expansion | 2.0 | 3.2 | 11 |
| 10 | Loyalty program | 0.8 | 1.3 | 6 |
| 11 | Fleet electrification | 1.9 | 2.1 | 5 |
| 12 | Analytics training | 0.4 | 0.7 | 5 |

1. **Formulate** in a markdown cell: one 0/1 variable per project, maximize total NPV, the budget, the staff-month limit, and the two rules as linear constraints.
2. **Build and solve** with `vtype=GRB.BINARY`. Print the chosen portfolio, total NPV, capital used and staff-months used.
3. **Re-solve with an $8M budget.** In 3–4 sentences: which projects fell out, which (if any) came in, and what that says about bang-for-buck. Were the ones that left the lowest NPV-per-dollar? They often are not — say why.

In [ ]:
projects = {  # id: (name, cost $M, NPV $M, staff-months)
    1:  ('Warehouse automation',       2.4, 3.9, 14),
    2:  ('Customer data platform',     1.8, 2.6,  9),
    3:  ('Core ERP upgrade',           3.1, 3.4, 16),
    4:  ('Mobile app relaunch',        0.9, 1.5,  7),
    5:  ('Boston retail expansion',    2.7, 4.1, 12),
    6:  ('Supplier portal',            0.6, 0.8,  4),
    7:  ('Demand forecasting engine',  1.2, 2.3,  8),
    8:  ('Energy retrofit',            1.5, 1.7,  3),
    9:  ('Hartford retail expansion',  2.0, 3.2, 11),
    10: ('Loyalty program',            0.8, 1.3,  6),
    11: ('Fleet electrification',      1.9, 2.1,  5),
    12: ('Analytics training',         0.4, 0.7,  5),
}
budget, staff_limit = 10.0, 60

hw = gp.Model('portfolio'); hw.Params.OutputFlag = 0

# TODO 1: one binary variable per project

# TODO 2: objective — maximize total NPV

# TODO 3: budget and staff-month constraints

# TODO 4: the two logical rules — 7 requires 3; 5 and 9 mutually exclusive

hw.optimize()

# TODO 5: print the chosen portfolio, total NPV, capital used, staff-months used

# TODO 6: change the budget constraint's RHS to 8.0, re-solve, and print what changed

# TODO 7: write your 3-4 sentences in a markdown cell below (and paste them into the Canvas text box)

---
### Submission checklist
- [ ] Formulation in a markdown cell, both logical rules written as inequalities
- [ ] Binary model solves; portfolio, NPV, capital and staff-months printed at $10M
- [ ] Re-solved at $8M; the difference printed, not eyeballed
- [ ] 3–4 sentences on bang-for-buck — pasted into the Canvas text box

**Next week — Session 7:** set covering, shift scheduling with Google's CP-SAT — and the midterm case launches. Join your team on Canvas (People → Groups → Teams) before then. 🗓️